# Evaluate UNet Segmentation with Skeleton Metrics

Runs `segmentation_skeleton_metrics.evaluate.evaluate(...)` on the same brain/segmentation pair used by `load_skeletons.ipynb`. Outputs a CSV report and a text summary in `metrics_out/`.

**Note:** This re-reads the SWCs from GCS (~15 min) because the metrics package builds its own labeled-graph representation that differs from `BrainDataset`. The `dataset_cache_*.pkl` does not help here.

### Imports

In [1]:
import os
import sys

from segmentation_skeleton_metrics.evaluate import evaluate
from segmentation_skeleton_metrics.utils.img_util import TensorStoreImage

# Shared dataset-config helper (segmentation-id lookup).
sys.path.insert(0, "../scripts")
from dataset_config import get_segmentation_id

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../configs/zihan_gcs_token.json"
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

### Paths

In [2]:
# Match load_skeletons.ipynb so we evaluate the same UNet checkpoint:
# segmentation id is looked up from configs/segmentation_datasets.rtf by brain_id.
brain_id = "794492"
segmentation_id = get_segmentation_id(brain_id)
print(f"brain_id={brain_id} -> segmentation_id={segmentation_id}")

gt_path = f"gs://allen-nd-goog/ground_truth_tracings/{brain_id}/voxel"
fragments_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/swcs"
segmentation_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/"

output_dir = f"../metrics_out/{brain_id}/{segmentation_id}"
os.makedirs(output_dir, exist_ok=True)

brain_id=794493 -> segmentation_id=mean40.stddev105.mask.136168199.no_omitted_20k.ffn.mt_0.1


### Run Evaluation

Three-step pipeline run by `evaluate(...)`:
1. **Label graphs** -- read GT SWCs and label each node with the UNet segment ID it falls inside (uses `segmentation`).
2. **Detect errors** -- per edge, classify as correct / split / omit / merge by comparing endpoint labels.
3. **Compute metrics** -- # Splits, # Merges, % Split/Omit/Merged Edges, ERL, Normalized ERL, Edge Accuracy, Split Rate, Merge Rate.

Saved artifacts: `results.csv` (per-GT-SWC), `results_overview.txt` (averaged + totals).

In [3]:
# anisotropy: same (x, y, z) microns/voxel as load_skeletons.ipynb. Not
#   applied to the GT SWCs because gt_path ends in /voxel (already in voxel
#   coords); use_anisotropy=False reflects that.
anisotropy = (0.748, 0.748, 1.0)

# swap_axes=True: SWC voxel coords are stored (x, y, z); after the
# `from_google` transpose the segmentation volume's last 3 dims are
# (z, y, x). Setting swap_axes adds an additional transpose so the image's
# axes line up with the SWC's (x, y, z) ordering. Without this you get
# IndexError: OUT_OF_RANGE on the first GT skeleton labeling pass.
segmentation = TensorStoreImage(segmentation_path, swap_axes=True)

evaluate(
    gt_path,
    segmentation,
    output_dir,
    anisotropy=anisotropy,
    fragments_path=fragments_path,
    use_anisotropy=False,
    save_merges=True,
    save_fragments=False,
    verbose=True,
)

I0609 15:41:33.737060 1982099 google_auth_provider.cc:149] Using credentials at ../configs/zihan_gcs_token.json
I0609 15:41:33.738329 1982099 google_auth_provider.cc:165] Using ServiceAccount AuthProvider



(1) Load Ground Truth


Label Graphs: 100%|██████████| 10/10 [01:45<00:00, 10.56s/it]



(2) Load Fragments


Build Graphs: 100%|██████████| 286564/286564 [07:57<00:00, 599.54it/s] 



(3) Compute Metrics


Added Cable Length (μm): 100%|██████████| 51/51 [00:53<00:00,  1.05s/it]



Average Results...
  # Splits: 1456.7007
  # Merges: 5.7246
  % Split Edges: 0.9120
  % Omit Edges: 3.3091
  % Merged Edges: 14.7613
  ERL: 1734.1010
  Normalized ERL: 0.0050
  Edge Accuracy: 81.0179
  Split Rate: 338.1301
  Merge Rate: 66409.2846

Total Results...
  # Splits: 12420
  # Merges: 51


### Inspect Results

In [4]:
import pandas as pd

results = pd.read_csv(os.path.join(output_dir, "results.csv"), index_col=0)
print(f"Per-SWC results ({len(results)} ground-truth skeletons):")
results

Per-SWC results (10 ground-truth skeletons):


,SWC Run Length,# Splits,# Merges,% Split Edges,% Omit Edges,% Merged Edges,ERL,Normalized ERL,Edge Accuracy,Split Rate,Merge Rate
N001-794493-SK,418564.952151,895.0,6,0.619330,0.632403,38.955016,1022.47,0.0024,59.79,461.82,68887.61
N002-794493-SP,112342.231182,981.0,2,2.138661,14.161043,0.379910,678.57,0.0060,83.32,95.85,47015.39
N003-794493-HP,361043.823735,1529.0,12,0.997688,5.562112,5.232279,956.87,0.0027,88.21,220.64,28113.34
N004-794493-SP,538457.033950,2822.0,10,1.439560,6.024648,6.396803,634.78,0.0012,86.14,176.56,49826.55
N005-794493-HP,396576.296446,926.0,0,0.753539,0.540627,0.000000,4141.19,0.0104,98.71,422.73,NaN
N007-794493-SP,564666.162581,2654.0,5,1.165711,4.971790,1.093230,1571.71,0.0028,92.77,199.70,106001.95
N008-794493-PP,392871.626980,793.0,5,0.540693,2.446902,20.859102,1640.52,0.0042,76.15,480.62,76226.84
N009-794493-HP,329789.382471,665.0,4,0.500936,0.741661,46.157937,2285.30,0.0069,52.60,489.76,81422.86
N010-794493-BP,194855.531690,395.0,0,0.486038,1.690923,0.000000,3805.44,0.0195,97.82,482.57,NaN
N011-794493-SP,300843.765097,760.0,7,0.757860,0.933809,25.170679,1326.82,0.0044,73.14,389.15,42250.64


In [5]:
with open(os.path.join(output_dir, "results_overview.txt")) as f:
    print(f.read())


Average Results...
  # Splits: 1456.7007
  # Merges: 5.7246
  % Split Edges: 0.9120
  % Omit Edges: 3.3091
  % Merged Edges: 14.7613
  ERL: 1734.1010
  Normalized ERL: 0.0050
  Edge Accuracy: 81.0179
  Split Rate: 338.1301
  Merge Rate: 66409.2846

Total Results...
  # Splits: 12420
  # Merges: 51

